# 05b — Teşhis: model neden val'de ~0.54 mAP50'de takılıyor?

**Çalışma zamanı: T4 GPU yeterli** (ucuz). Eğitim yok, sadece tahmin. Toplam ~15 dk, çoğu dataset'i açmak.

E01 (640) ve E02 (1280) val'de aynı tavana takıldı. İki olası sebep var ve çözümleri tamamen farklı:

| Hipotez | Ne olur | Ölçüt |
|---|---|---|
| **H1 — etiket bölünmesi** | Model ateşi buluyor ama alevi etiketçiden farklı sayıda kutuya bölüyor; IoU 0.5'te kaçırma sayılıyor | IoU 0.1'de recall ≫ IoU 0.5'te recall · piksel kapsama yüksek, kutu recall düşük |
| **H2 — sahne ezberi** | Model train sahnelerini öğreniyor, yeni sahnelere genellemiyor | train örneğinde fire mAP ≫ val'de fire mAP |

Ek olarak: görüntü düzeyinde tespit oranı ("bu karede ateş var mı?" — gerçek kullanımda asıl soru), kutu boyuna göre recall, sahne bazında hatalar.

**Test seti burada da açılmıyor.** Sadece train ve val.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!pip -q install ultralytics==8.4.183

In [ ]:
import os, json, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
from PIL import Image
from ultralytics import YOLO

ROOT   = Path('/content/drive/MyDrive/firesmoke')
REP    = ROOT / 'reports'
RUNS   = ROOT / 'runs'

# --- Ayarlar -------------------------------------------------------------------
EXP     = 'E01_y11s_640'   # incelenecek deney (E02_y11s_1280 için de tekrar çalıştırılabilir)
CONF    = 0.25             # "tespit" sayılacak en düşük güven
N_TRAIN = 3000             # train'den rastgele örnek (H2 karşılaştırması için)
SEED    = 0
DSET    = 'firesmoke_v2'   # val her sürümde aynı; dataset'i açmak için
# ------------------------------------------------------------------------------

DS_ZIP = ROOT / 'datasets' / f'{DSET}.zip'
DATA   = Path('/content') / DSET

args  = yaml.safe_load((RUNS / EXP / 'args.yaml').read_text())
IMGSZ = int(args['imgsz'])
BEST  = RUNS / EXP / 'weights' / 'best.pt'
WORKERS = min(8, os.cpu_count() or 2)
OUTD = REP / f'diag_{EXP}'
OUTD.mkdir(parents=True, exist_ok=True)
NAMES = {0: 'fire', 1: 'smoke'}
print(EXP, '· imgsz', IMGSZ, '·', BEST.exists() and 'best.pt var')

## 1. Dataset'i aç

In [ ]:
t0 = time.time()
if not (DATA / 'data.yaml').exists():
    subprocess.run(['unzip', '-q', '-o', str(DS_ZIP), '-d', str(DATA.parent)], check=True)
man = pd.read_csv(DATA / 'manifest.csv')
val = man[man.split == 'val'].reset_index(drop=True)
trs = man[man.split == 'train'].sample(min(N_TRAIN, int((man.split == 'train').sum())), random_state=SEED).reset_index(drop=True)
print(f'val {len(val)} · train örneği {len(trs)}  ({time.time() - t0:.0f} sn)')

## 2. H2 — train örneği ile val arasındaki fark (Ultralytics mAP)

Aynı model, aynı ayarlar; bir kez train'den rastgele 3000 görüntüde, bir kez val'de ölçülür. Model train sahnelerini ezberlediyse train'de çok daha yüksek çıkar.

In [ ]:
model = YOLO(str(BEST))

def ul_val(df, tag):
    lst = DATA / f'_diag_{tag}.txt'
    lst.write_text('\n'.join(str(DATA / f) for f in df.file) + '\n')
    y = DATA / f'_diag_{tag}.yaml'
    y.write_text(f'path: {DATA}\ntrain: {lst}\nval: {lst}\nnames:\n  0: fire\n  1: smoke\n')
    r = model.val(data=str(y), imgsz=IMGSZ, batch=16, workers=WORKERS, plots=False, verbose=False,
                  project=str(OUTD), name=f'ul_{tag}', exist_ok=True)
    rows = []
    for j, c in enumerate(r.box.ap_class_index):
        rows.append({'küme': tag, 'sınıf': NAMES[int(c)], 'P': r.box.p[j], 'R': r.box.r[j],
                     'mAP50': r.box.ap50[j], 'mAP50-95': r.box.ap[j]})
    return rows

h2 = pd.DataFrame(ul_val(trs, 'train_örnek') + ul_val(val, 'val')).round(3)
display(h2)

## 3. Tahminler

Her görüntü için tahminler bir kez alınır (conf ≥ 0.05), sonraki tüm ölçümler bunları kullanır.

In [ ]:
def read_gt(df):
    gt = {}
    for uid, f in zip(df.uid, df.file):
        p = DATA / f.replace('images/', 'labels/', 1).rsplit('.', 1)[0]
        p = Path(str(p) + '.txt')
        a = np.loadtxt(p, ndmin=2) if p.exists() and p.stat().st_size else np.zeros((0, 5))
        c, x, y, w, h = a.T if len(a) else (np.zeros(0),) * 5
        gt[uid] = (c.astype(int), np.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], 1) if len(a) else np.zeros((0, 4)))
    return gt

import gc, torch
gc.collect(); torch.cuda.empty_cache()

def predict(df, bs=32):
    # Ultralytics liste kaynağını tek batch yapıyor (batch= yok sayılıyor) → listeyi elle böl
    files, out, t0 = [str(DATA / f) for f in df.file], {}, time.time()
    for s in range(0, len(files), bs):
        for r in model.predict(source=files[s:s + bs], imgsz=IMGSZ, conf=0.05, verbose=False):
            b = r.boxes
            out[Path(r.path).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(),
                                      b.xyxyn.cpu().numpy(), r.orig_shape)
    print(f'{len(out)} görüntü · {time.time() - t0:.0f} sn')
    return out

GT = {'val': read_gt(val), 'train_örnek': read_gt(trs)}
PR = {'val': predict(val), 'train_örnek': predict(trs)}
DF = {'val': val, 'train_örnek': trs}

## 4. H1 — IoU eşiğine göre recall/precision ve piksel kapsama

- **Kutu eşleşmesi:** her tahmin en fazla bir etiketle eşleşir. IoU 0.5 standart; 0.1 "aynı yeri gösteriyor mu" demek.
- **Piksel kapsama:** etiket kutularının birleşimi ile tahmin kutularının birleşimi karşılaştırılır. Kutuların nasıl bölündüğünden bağımsızdır: "ateşli bölgenin ne kadarını işaretledi?"

IoU 0.1 recall'ı IoU 0.5'ten çok yüksekse ya da piksel recall'ı kutu recall'ından çok yüksekse **H1** geçerli.

In [ ]:
def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-12)

def match(g, p, thr):
    """p güvene göre sıralı. Dönen: eşleşen etiket maskesi, eşleşen tahmin maskesi."""
    mg, mp = np.zeros(len(g), bool), np.zeros(len(p), bool)
    if len(g) and len(p):
        M = iou(p, g)
        for i in range(len(p)):
            cand = np.where(~mg & (M[i] >= thr))[0]
            if len(cand):
                j = cand[np.argmax(M[i, cand])]; mg[j] = mp[i] = True
    return mg, mp

GRID = 128
def raster(boxes):
    m = np.zeros((GRID, GRID), bool)
    for x1, y1, x2, y2 in np.clip(boxes, 0, 1):
        m[int(y1 * GRID):max(int(np.ceil(y2 * GRID)), int(y1 * GRID) + 1),
          int(x1 * GRID):max(int(np.ceil(x2 * GRID)), int(x1 * GRID) + 1)] = True
    return m

def per_image(tag):
    rows = []
    for uid in DF[tag].uid:
        gc, gb = GT[tag][uid]
        pc, pconf, pb, (H, W) = PR[tag][uid]
        keep = pconf >= CONF
        pc, pconf, pb = pc[keep], pconf[keep], pb[keep]
        o = np.argsort(-pconf); pc, pb = pc[o], pb[o]
        for c in (0, 1):
            g, p = gb[gc == c], pb[pc == c]
            r = {'uid': uid, 'cls': NAMES[c], 'n_gt': len(g), 'n_pr': len(p)}
            for thr in (0.1, 0.25, 0.5):
                mg, mp = match(g, p, thr)
                r[f'tp_gt@{thr}'], r[f'tp_pr@{thr}'] = int(mg.sum()), int(mp.sum())
            mg5, _ = match(g, p, 0.5); mg1, _ = match(g, p, 0.1)
            r['gt_px'] = [float(np.sqrt((b[2] - b[0]) * W * (b[3] - b[1]) * H) * IMGSZ / max(H, W)) for b in g]
            r['gt_hit@0.5'], r['gt_hit@0.1'] = mg5.tolist(), mg1.tolist()
            A, B = raster(g), raster(p)
            r['px_gt'], r['px_pr'], r['px_int'] = int(A.sum()), int(B.sum()), int((A & B).sum())
            rows.append(r)
    return pd.DataFrame(rows)

PI = {t: per_image(t) for t in DF}

def box_table(tag):
    out = []
    for c, d in PI[tag].groupby('cls'):
        r = {'küme': tag, 'sınıf': c}
        for thr in (0.1, 0.25, 0.5):
            r[f'R@{thr}'] = d[f'tp_gt@{thr}'].sum() / max(d.n_gt.sum(), 1)
            r[f'P@{thr}'] = d[f'tp_pr@{thr}'].sum() / max(d.n_pr.sum(), 1)
        r['piksel R'] = d.px_int.sum() / max(d.px_gt.sum(), 1)
        r['piksel P'] = d.px_int.sum() / max(d.px_pr.sum(), 1)
        out.append(r)
    return out

h1 = pd.DataFrame(box_table('val') + box_table('train_örnek')).round(3)
display(h1)

## 5. Görüntü düzeyinde tespit — "bu karede ateş/duman var mı?"

Gerçek bir yangın gözetleme sisteminde asıl soru budur. Bir karede o sınıftan en az bir etiket varken en az bir tahmin (conf ≥ `CONF`) varsa **yakalandı**; hiç etiket yokken tahmin varsa **yanlış alarm**.

In [ ]:
def image_level(tag):
    d = PI[tag].merge(DF[tag][['uid', 'source', 'cat']], on='uid')
    rows = []
    for (c, src), g in list(d.groupby(['cls', 'source'])) + [((c, 'hepsi'), g) for c, g in d.groupby('cls')]:
        pos, neg = g[g.n_gt > 0], g[g.n_gt == 0]
        rows.append({'küme': tag, 'sınıf': c, 'kaynak': src,
                     'pozitif kare': len(pos), 'yakalama %': 100 * (pos.n_pr > 0).mean() if len(pos) else np.nan,
                     'boş kare': len(neg), 'yanlış alarm %': 100 * (neg.n_pr > 0).mean() if len(neg) else np.nan,
                     'tamamen negatif karede yanlış alarm %':
                         100 * (g[g.cat == 'negatif'].n_pr > 0).mean() if (g.cat == 'negatif').any() else np.nan})
    return rows

il = pd.DataFrame(image_level('val') + image_level('train_örnek')).round(1)
display(il)

## 6. Kutu boyuna göre recall (val)

Kutu boyu = eğitimdeki ölçekte (letterbox sonrası) √(genişlik × yükseklik) piksel. Küçük kutularda IoU 0.5 çok sert: 8 px'lik kutuda 2 px kayma bile eşleşmeyi bozar.

In [ ]:
BINS, LABS = [0, 8, 16, 32, 64, 1e9], ['<8', '8–16', '16–32', '32–64', '≥64']
rows = []
for c, d in PI['val'].groupby('cls'):
    px  = np.concatenate([np.array(x, float) for x in d.gt_px]) if len(d) else np.zeros(0)
    h5  = np.concatenate([np.array(x, bool) for x in d['gt_hit@0.5']]) if len(d) else np.zeros(0, bool)
    h1_ = np.concatenate([np.array(x, bool) for x in d['gt_hit@0.1']]) if len(d) else np.zeros(0, bool)
    b = pd.cut(px, BINS, labels=LABS, right=False)
    for lab in LABS:
        m = np.asarray(b == lab)
        rows.append({'sınıf': c, 'boy (px)': lab, 'etiket': int(m.sum()), 'pay %': 100 * m.mean() if len(m) else 0,
                     'R@0.5': h5[m].mean() if m.any() else np.nan, 'R@0.1': h1_[m].mean() if m.any() else np.nan})
size_tab = pd.DataFrame(rows).round(3)
display(size_tab)

## 7. Sahne bazında (val, fire)

Val'deki fire karelerinin sahnelere (grup) dağılımı ve her sahnede görüntü düzeyinde yakalama oranı. Hataların birkaç sahnede toplanması, o sahnelerin train'de benzerinin olmadığını gösterir (H2'ye destek).

In [ ]:
d = PI['val'].merge(val[['uid', 'group', 'source']], on='uid')
f = d[(d.cls == 'fire') & (d.n_gt > 0)]
scene = (f.groupby('group')
          .agg(kare=('uid', 'size'), etiket=('n_gt', 'sum'), yakalama=('n_pr', lambda s: 100 * (s > 0).mean()),
               R05=('tp_gt@0.5', 'sum'))
          .assign(**{'R@0.5 %': lambda t: 100 * t.R05 / t.etiket}).drop(columns='R05')
          .sort_values('kare', ascending=False).round(1))
print(f'val fire kareleri {len(f)} kare · {len(scene)} sahnede; en büyük 5 sahne karelerin %{100 * scene.kare.head(5).sum() / len(f):.0f}\'i')
display(scene.head(15))

## 8. Etiket yoğunluğu ve örnekler

Fire içeren karelerde kare başına kutu sayısı (train ve val). Ardından iki grid:
1. **Kutu recall'ı düşük ama piksel kapsaması yüksek** kareler: model ateşi bulmuş, kutuları farklı bölmüş (H1 örnekleri).
2. **Ateşi tamamen kaçırdığı** kareler.

Yeşil: etiket · kırmızı: tahmin (conf ≥ `CONF`).

In [ ]:
dens = []
for s in ('train', 'val'):
    n = man[(man.split == s) & (man.n_fire > 0)].n_fire
    dens.append({'split': s, 'fire kare': len(n), 'ortalama kutu': n.mean(), 'medyan': n.median(),
                 '≥10 kutulu kare %': 100 * (n >= 10).mean(), 'tek kutulu kare %': 100 * (n == 1).mean()})
display(pd.DataFrame(dens).round(2))

def show(uids, title, fname):
    if not len(uids):
        print('örnek yok:', title); return
    fig, axes = plt.subplots((len(uids) + 1) // 2, 2, figsize=(12, 3.6 * ((len(uids) + 1) // 2)), squeeze=False)
    for ax, uid in zip(axes.ravel(), uids):
        fpath = DATA / val.set_index('uid').file[uid]
        with Image.open(fpath) as im:
            W, H = im.size; ax.imshow(im.convert('RGB'))
        gc, gb = GT['val'][uid]; pc, pconf, pb, _ = PR['val'][uid]
        for b in gb[gc == 0]:
            ax.add_patch(patches.Rectangle((b[0] * W, b[1] * H), (b[2] - b[0]) * W, (b[3] - b[1]) * H, fill=False, ec='lime', lw=1.2))
        for b in pb[(pc == 0) & (pconf >= CONF)]:
            ax.add_patch(patches.Rectangle((b[0] * W, b[1] * H), (b[2] - b[0]) * W, (b[3] - b[1]) * H, fill=False, ec='red', lw=1.2, ls='--'))
        r = fv.set_index('uid').loc[uid]
        ax.set_title(f'{uid[:30]} · etiket {r.n_gt} · tahmin {r.n_pr} · R@0.5 {r["tp_gt@0.5"]}/{r.n_gt} · piksel R {r.pxr:.2f}', fontsize=7)
        ax.axis('off')
    for ax in axes.ravel()[len(uids):]:
        ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(OUTD / fname, dpi=90); plt.show()

fv = PI['val'][(PI['val'].cls == 'fire') & (PI['val'].n_gt > 0)].copy()
fv['r05'] = fv['tp_gt@0.5'] / fv.n_gt
fv['pxr'] = fv.px_int / fv.px_gt.clip(lower=1)
rng = np.random.default_rng(SEED)
split_ex = fv[(fv.pxr >= 0.5) & (fv.r05 <= 0.34) & (fv.n_gt >= 2)].uid.to_numpy()
miss_ex  = fv[fv.n_pr == 0].uid.to_numpy()
show(rng.permutation(split_ex)[:8], 'Bölge bulunmuş, kutular farklı bölünmüş (H1)', 'grid_split.png')
show(rng.permutation(miss_ex)[:6], 'Ateş hiç bulunamamış', 'grid_miss.png')
print(f'H1 tipi kare: {len(split_ex)} / {len(fv)} · tamamen kaçırılan: {len(miss_ex)} / {len(fv)}')

## 9. Özet — bunu bana gönder

In [ ]:
def g(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items():
        m &= (df[k] == v).to_numpy()
    return df[m].iloc[0]

L = [f'# Teşhis — {EXP} (imgsz {IMGSZ}, conf {CONF})', '']
for c in ('fire', 'smoke'):
    a, b = g(h2, küme='train_örnek', sınıf=c), g(h2, küme='val', sınıf=c)
    v, t = g(h1, küme='val', sınıf=c), g(h1, küme='train_örnek', sınıf=c)
    i = g(il, küme='val', sınıf=c, kaynak='hepsi')
    L += [f'## {c}',
          f'- H2 mAP50: train örneği {a.mAP50:.3f} · val {b.mAP50:.3f}  (mAP50-95: {a["mAP50-95"]:.3f} · {b["mAP50-95"]:.3f})',
          f'- H1 val recall: IoU0.1 {v["R@0.1"]:.2f} · IoU0.25 {v["R@0.25"]:.2f} · IoU0.5 {v["R@0.5"]:.2f} · piksel {v["piksel R"]:.2f}',
          f'- H1 val precision: IoU0.1 {v["P@0.1"]:.2f} · IoU0.5 {v["P@0.5"]:.2f} · piksel {v["piksel P"]:.2f}',
          f'- train örneği recall: IoU0.1 {t["R@0.1"]:.2f} · IoU0.5 {t["R@0.5"]:.2f} · piksel {t["piksel R"]:.2f}',
          f'- Görüntü düzeyi (val): yakalama %{i["yakalama %"]:.1f} · yanlış alarm %{i["yanlış alarm %"]:.1f} · '
          f'tamamen negatif karede yanlış alarm %{i["tamamen negatif karede yanlış alarm %"]:.1f}', '']
L += ['## val, kutu boyuna göre recall', size_tab.to_markdown(index=False), '',
      '## val fire, en büyük 10 sahne', scene.head(10).to_markdown(), '',
      '## Etiket yoğunluğu', pd.DataFrame(dens).round(2).to_markdown(index=False), '',
      f'H1 tipi kare: {len(split_ex)} / {len(fv)} · tamamen kaçırılan: {len(miss_ex)} / {len(fv)}']
txt = '\n'.join(L)
(OUTD / 'ozet.md').write_text(txt)
print(txt)

# deney karşılaştırma günlüğü: mAP'nin yanında sağlam ölçütler
dsc = PI['val'].merge(val[['uid', 'group']], on='uid')
row = {'tarih': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'), 'EXP': EXP, 'imgsz': IMGSZ, 'conf': CONF}
for c in ('fire', 'smoke'):
    v, i = g(h1, küme='val', sınıf=c), g(il, küme='val', sınıf=c, kaynak='hepsi')
    pos = dsc[(dsc.cls == c) & (dsc.n_gt > 0)]
    row.update({f'{c}_mAP50_val': g(h2, küme='val', sınıf=c).mAP50,
                f'{c}_mAP50_train': g(h2, küme='train_örnek', sınıf=c).mAP50,
                f'{c}_kare_yakalama': i['yakalama %'], f'{c}_yanlis_alarm': i['yanlış alarm %'],
                f'{c}_sahne_ort_yakalama': round(100 * pos.groupby('group').n_pr.apply(lambda s: (s > 0).mean()).mean(), 1),
                f'{c}_piksel_R': v['piksel R'], f'{c}_piksel_P': v['piksel P'], f'{c}_R@0.5': v['R@0.5']})
logd = REP / 'experiments_diag.csv'
dl = pd.concat([pd.read_csv(logd), pd.DataFrame([row])], ignore_index=True) if logd.exists() else pd.DataFrame([row])
dl = dl.drop_duplicates(subset=['EXP', 'conf'], keep='last')
dl.to_csv(logd, index=False)
display(dl.set_index('EXP').T)

**Bana gönderilecekler:** son hücrenin çıktısı (özet + deney karşılaştırma tablosu `reports/experiments_diag.csv`) (`reports/diag_<EXP>/ozet.md` olarak da kaydedilir) ve iki grid hakkında bir cümle: kırmızı kutular ateşin üstünde mi, sadece farklı mı bölünmüş?